Sequential Chats and Customer Onboarding

## Setup

In [1]:
#from utils import get_openai_api_key
#OPENAI_API_KEY = get_openai_api_key()
#print(OPENAI_API_KEY)
#llm_config = {"model": "gpt-3.5-turbo","api_key":OPENAI_API_KEY}

In [2]:
llm_config = {"model": "gemini-3.5-flash","api_key":"","api_type":"google"}

In [3]:
from autogen import ConversableAgent

C:\Users\User\anaconda3\Lib\site-packages\autogen\oai\gemini.py:65: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Patching name='__init__', member=<function BedrockClient.__init__ at 0x000001F23AF5E520>, patched=<function function.__call__ at 0x000001F23AF5E5C0>
Patching name='_retries', member=5, patched=5
Patching name='cost', member=<function BedrockClient.cost at 0x000001F23AF5E8E0>, patched=<function function.__call__ at 0x000001F23AF5EB60>
Patching name='create', member=<function BedrockClient.create at 0x000001F23AF5E840>, patched=<function function.__call__ at 0x000001F23AF5EC00>
Patching name='get_usage', member=<function BedrockClient.get_usage at 0x000001F23AF5E980>, patched=<function function.__call__ at 0x000001F23AF5ECA0>
Patching name='message_retrieval', member=<function BedrockClient.message_retrieval at 0x000001F23AF5E660>, patched=<function function.__call__ at 0x000001F23AF5ED40>
Patching name='parse_custom_params', member=<function BedrockClient.parse_custom_params at 0x000001F23AF5E700>, patched=<function function.__call__ at 0x000001F23AF5EDE0>
Patching name='parse_params', 

## Creating the needed agents

In [4]:
#from utils import get_openai_api_key
#OPENAI_API_KEY = get_openai_api_key()

#llm_config = {"model": "gpt-3.5-turbo", "api_key": OPENAI_API_KEY}

from autogen import ConversableAgent

onboarding_personal_information_agent = ConversableAgent(
    name="onboarding_personal_information_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's name and location.
    Do not ask for other information. Return 'TERMINATE' 
    when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

onboarding_topic_preference_agent = ConversableAgent(
    name="onboarding_topic_preference_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's preferences on news topics.
    Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

customer_engagement_agent = ConversableAgent(
    name="customer_engagement_agent",  # Removed spaces
    system_message='''You are a helpful customer service agent.
    You are here to provide fun for the customer based on the user's
    personal information and topic preferences.
    This could include fun facts, jokes, or interesting stories.
    Make sure to make it engaging and fun!
    Return 'TERMINATE' when you are done.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)

customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)


## Creating tasks

Now, you can craft a series of tasks to facilitate the onboarding process.

In [5]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": "Hello, I'm here to help you get started with our product. Could you tell me your name and location?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return the customer information as JSON object only: {'name': '', 'location': ''}",
        },
        "max_turns": 1,
        "clear_history": True
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message": "Great! Could you tell me what topics you are interested in reading about?",
        "summary_method": "reflection_with_llm",
        "max_turns": 1,
        "clear_history": False
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1,
        "summary_method": "reflection_with_llm",
    },
]

## Start the onboarding process

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different inputs, such as name, location, and preferences.

In [6]:
from autogen import initiate_chats

chat_results = initiate_chats(chats)



********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_personal_information_agent (to customer_proxy_agent):

Hello, I'm here to help you get started with our product. Could you tell me your name and location?

--------------------------------------------------------------------------------


C:\Users\User\anaconda3\Lib\site-packages\autogen\agentchat\chat.py:51: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(


Replying as customer_proxy_agent. Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Smila from Dindigul


customer_proxy_agent (to onboarding_personal_information_agent):

Smila from Dindigul

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_topic_preference_agent (to customer_proxy_agent):

Great! Could you tell me what topics you are interested in reading about?
Context: 
{"name": "Smila", "location": "Dindigul"}

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  DataScience


customer_proxy_agent (to onboarding_topic_preference_agent):

DataScience

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
customer_proxy_agent (to customer_engagement_agent):

Let's find something fun to read.
Context: 
{"name": "Smila", "location": "Dindigul"}
Smila from Dindigul is interested in reading about Data Science.

--------------------------------------------------------------------------------
customer_engagement_agent (to customer_proxy_agent):

Vanakkam Smila! Welcome! It is absolutely wonderful to meet you. A big hello to you all the way in beautiful Dindigul! 

Since Dindigul is world-famous for its legendary, mouth-watering Biryani, I was just thinking... imagine if we used **Data Science** to craft the *ultimate* Biryani recipe? We could run a regres

## Print out the summary

In [7]:
summary=[]
for chat_result in chat_results:
    print(chat_result.summary)
    summary.append(chat_result.summary)
    print("\n")

{'content': '{"name": "Smila", "location": "Dindigul"}', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


{'content': 'Smila from Dindigul is interested in reading about Data Science.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


{'content': 'Smila from Dindigul, who is interested in Data Science, received a personalized and engaging overview of the field. The key takeaways highlighted the massive scale of global data growth, a real-world example of how Netflix uses predictive data science to guarantee successful content, and the playful application of data concepts to local culture and humor.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}




In [8]:
summary[0]

{'content': '{"name": "Smila", "location": "Dindigul"}',
 'refusal': None,
 'role': 'assistant',
 'annotations': None,
 'audio': None,
 'function_call': None,
 'tool_calls': None}

In [9]:
#!pip install pandas
#%pip install ast

In [10]:
import pandas as pd
import ast 

# Input string
#data_str = "{'name': 'Ramisha', 'location': 'Coimbatore'}"
data_str=summary[0]["content"]


# Convert string to dictionary
data_dict = ast.literal_eval(data_str)

# Convert dictionary to DataFrame
df = pd.DataFrame([data_dict])

df


,name,location
0,Smila,Dindigul


In [11]:
import ast
print(ast.__name__)


ast


In [12]:
df["TopicPreference"]=summary[1]["content"]

In [13]:
df

,name,location,TopicPreference
0,Smila,Dindigul,Smila from Dindigul is interested in reading a...


In [14]:
df["Engagement"]=summary[2]["content"]

In [15]:
df

,name,location,TopicPreference,Engagement
0,Smila,Dindigul,Smila from Dindigul is interested in reading a...,"Smila from Dindigul, who is interested in Data..."


In [16]:
#%pip install gspread
#%pip install oauth2client

In [17]:
def add_leads_to_google_sheet(df):
    import gspread
    from oauth2client.service_account import ServiceAccountCredentials

    # Define the scope for Google Sheets API
    scope = ['https://www.googleapis.com/auth/spreadsheets']

    # Add credentials to the account
    creds = ServiceAccountCredentials.from_json_keyfile_name(
        'websiteleadchatgptapi-a46f50ae1507.json', scope
    )

    # Authorize the client
    client = gspread.authorize(creds)

    # Specify the Google Sheet ID
    sheet_id = '1w-19wp0W2nRV5Pde5gim6kKlCoPTjv6mT3A8lAdNrcM'

    # Open the spreadsheet using its ID
    spreadsheet = client.open_by_key(sheet_id)

    # Select the first worksheet (index 0)
    worksheet = spreadsheet.get_worksheet(0)

    # Append each row of the DataFrame to the worksheet
    for _, row in df.iterrows():
        # Convert row to a list and append to the sheet
        worksheet.append_row(row.tolist())

    print("Data added successfully!")
    return "added"


In [20]:
add_leads_to_google_sheet(df)

## Print out the cost

In [19]:
for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.5-flash': {'cost': 0, 'prompt_tokens': 52, 'completion_tokens': 16, 'total_tokens': 68}}, 'usage_excluding_cached_inference': {'total_cost': 0}}


{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.5-flash': {'cost': 0, 'prompt_tokens': 56, 'completion_tokens': 15, 'total_tokens': 71}}, 'usage_excluding_cached_inference': {'total_cost': 0}}


{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.5-flash': {'cost': 0, 'prompt_tokens': 706, 'completion_tokens': 591, 'total_tokens': 1297}}, 'usage_excluding_cached_inference': {'total_cost': 0}}




In [21]:
df

,name,location,TopicPreference,Engagement
0,Smila,Dindigul,Smila from Dindigul is interested in reading a...,"Smila from Dindigul, who is interested in Data..."
